In [4]:
import numpy as np
from tqdm import tqdm
from functools import partial
from concurrent.futures import  as_completed, ProcessPoolExecutor
import pandas as pd

# Classes

In [5]:
class k_armed_bandit:
    def __init__(self, k: int, reward_init: np.array, reward_drift: callable):
        '''
        K is the number of actions 
        `reward_init` is the starting reward value for each action. It must be a list of len k
        `reward_drift` is how much the reward value should change each step. It must be a function that takes a reward value and returns the new value.
        '''
        self.k = k
        self.rewards = reward_init
        self.reward_drift = reward_drift

        if len(self.rewards) != k:
            raise ValueError("The length of reward_init is not the correct length.")

    def step(self, action_taken) -> float:
        '''
        This function will return the value of an action taken. It will then conduct a step which may change the reward values of the actions  
        `action_taken` the action to take (zero-indexed), it will return the value of that action prior to step changes
        '''
        
        action_reward = np.random.normal(self.rewards[action_taken], 1)

        self.rewards = np.array([self.reward_drift(reward) for reward in self.rewards])

        return action_reward
    
    def is_optimal(self, action):
        '''
        This function will return if a particular action is the optimal action
        '''

        return action == self.rewards.argmax()

In [18]:
class bandit_learning_agent:
    def __init__(self, k: int, bandit: k_armed_bandit, epsilon: float, action_value_method: callable):
        '''
        `k` is the number of actions that can be taken
        `epsilon` is the proportion of exploratory actions
        `action_value_method` is how the agent determines its value for an action.
        '''
        self.bandit = bandit
        self.q = np.zeros(k)
        self.n = np.zeros(k)

        self.epsilon = epsilon

        self.action_value_method = action_value_method

        self.rewards_received = []
        self.optimal_action = []

    
    def act(self):
        # Are we taking a exploratory move or exploitive move?

        if np.random.random() > (1-self.epsilon):
            # Take an exploratory move
            action = np.random.randint(0, len(self.q)-1)
            # print("Making a random action")

        else:
            action = self.q.argmax()

        # print(f"Taking action {action} with value {self.q[action]}")

        self.optimal_action.append(int(self.bandit.is_optimal(action)))

        reward = self.bandit.step(action)

        # print(f"Recieved reward {reward}")

        self.rewards_received.append(reward)

        self.n[action] += 1
        self.q[action] = self.action_value_method(current = self.q[action], n = self.n[action], reward = reward)

        # print(f"New action value {self.q[action]}")

# Doing tests

In [7]:
def constant_bandit_generator(k):
    return k_armed_bandit(k, np.random.randn(k), lambda r: r)

def drift_bandit_generator(k):
    return k_armed_bandit(k, np.zeros(k), lambda r: r + np.random.normal(0, 0.01))

In [8]:
def increment(current, n, reward):
    return current + (1/n)*(reward-current)

def constant_step_size(a, current, n, reward):
    return current + a*(reward-current)

def learning_agent_generator(k, bandit, action_value_method, epsilon=0.1):
    return bandit_learning_agent(k, bandit, epsilon, action_value_method)

In [9]:
def bandit_learning_run(bandit_generator, agent_generator, steps = 10_000):
    bandit = bandit_generator()

    agent = agent_generator(bandit=bandit)

    for step in range(steps):
        agent.act()

    return np.array(agent.rewards_received), np.array(agent.optimal_action)

def k_armed_bandit_test_bed(bandit_generator, agent_generator, runs, steps):
    rewards = []
    optimal_action = []
    with ProcessPoolExecutor(max_workers=8) as executor:
        runs = {
            executor.submit(bandit_learning_run, bandit_generator, agent_generator, steps): i
            for i in range(runs)
        }
        for run in tqdm(as_completed(runs), total=len(runs)):
            run_reward, run_optimal_action = run.result()

            rewards.append(run_reward)
            optimal_action.append(run_optimal_action)

    return np.array(rewards), np.array(optimal_action)

# Exercise 2.5

In [ ]:
k = 10

learning_agents = {
    "increment": partial(learning_agent_generator, k=k, action_value_method = increment),
    "constant_step_size": partial(learning_agent_generator, k=k, action_value_method = partial(constant_step_size, a=0.1)),
}

results = {}

for name, generator in learning_agents.items():
    print(f"Doing constant reward problem {name}")
    constant_reward, constant_optimal_action = k_armed_bandit_test_bed(
        partial(constant_bandit_generator, k),
        generator,
        2_000,
        10_000
    )

    print(f"Doing drift problem with {name}")

    drift_reward, drift_optimal_action = k_armed_bandit_test_bed(
        partial(drift_bandit_generator, k),
        generator,
        2_000,
        10_000
    )

    results[name] = {
        "constant": {
            "rewards": constant_reward,
            "optimal_action": constant_optimal_action
        },
        "drift": {
            "rewards": drift_reward,
            "optimal_action": drift_optimal_action
        }
    }

Doing constant reward problem increment


 17%|█▋        | 331/2000 [00:13<01:08, 24.40it/s]


: 

: 

# Analyzing tests

In [ ]:
test_bed_results = results

In [ ]:
results

{'increment': {'constant': {'rewards': array([[ 0.21751272,  0.96331294, -1.19249089, ...,  2.36132674,
            0.74572061,  1.40465193],
          [ 0.21751272,  0.96331294, -1.19249089, ...,  2.36132674,
            0.74572061,  1.40465193],
          [ 0.21751272,  0.96331294, -1.19249089, ...,  2.36132674,
            0.74572061,  1.40465193],
          ...,
          [-2.41545506,  2.16286327,  1.20935038, ...,  2.14260915,
            0.2968967 ,  1.38792485],
          [ 2.51839103,  2.69652373,  2.53791805, ...,  0.81400029,
            0.95523364,  3.31687428],
          [ 0.6102377 ,  1.81209818,  1.38933585, ...,  0.16282968,
            3.17317929, -2.60070511]]),
   'optimal_action': array([[0., 0., 0., ..., 1., 1., 1.],
          [0., 0., 0., ..., 1., 1., 1.],
          [0., 0., 0., ..., 1., 1., 1.],
          ...,
          [0., 1., 1., ..., 1., 1., 1.],
          [1., 1., 1., ..., 1., 1., 1.],
          [0., 0., 0., ..., 1., 1., 0.]])},
  'drift': {'rewards': array(

In [ ]:
results_flattened = {
    f"{agent}_{bandit}": results
    for agent, bandits in results.items()
    for bandit, results in bandits.items()
}

results_flattened

results_averaged = {
    agent: {
        "avg_reward": results["rewards"].mean(0),
        "optimal_action_prop": results["optimal_action"].mean(0)
    }
    for agent, results in results_flattened.items()
}

results_averaged

dfs = []

for run, values in results_averaged.items():
    dfs.append(pd.DataFrame(
        values, columns=["avg_reward", "optimal_action_prop"]
    ).assign(run_name=run))

df = pd.concat(dfs)
df.reset_index(inplace=True)

df


,index,avg_reward,optimal_action_prop,run_name
0,0,-0.039585,0.0695,increment_constant
1,1,0.369891,0.1365,increment_constant
2,2,0.396396,0.1725,increment_constant
3,3,0.501515,0.2290,increment_constant
4,4,0.646491,0.2440,increment_constant
...,...,...,...,...
39995,9995,1.432704,0.7385,constant_step_size_drift
39996,9996,1.492871,0.7425,constant_step_size_drift
39997,9997,1.274522,0.7245,constant_step_size_drift
39998,9998,1.376170,0.7275,constant_step_size_drift


In [ ]:
import plotly.express as px

fig = px.scatter(df, y="avg_reward", x="index", color="run_name", symbol="run_name")
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
fig = px.scatter(df, y="optimal_action_prop", x="index", color="run_name", symbol="run_name")
fig.show()

# Exercise 2.10

I need to run a tests of 200,000 steps where the average reward is that of the last 100,000 steps.

I need to produce a paramter search graph for sample average and contstant step size with a=0.1. They are both going to be greedy algorithm and we are searching the space of e.

In [19]:
def non_stationary_bandit_generator():
    return k_armed_bandit(10, np.zeros(10), lambda r: r + np.random.normal(0, 0.01))

In [20]:
epsilon_range = [2 ** (-e) for e in reversed(range(1,10))]
epsilon_range

[0.001953125,
 0.00390625,
 0.0078125,
 0.015625,
 0.03125,
 0.0625,
 0.125,
 0.25,
 0.5]

In [21]:
k = 10

learning_agents = {
    "increment": partial(learning_agent_generator, k=k, action_value_method = increment),
    "constant_step_size": partial(learning_agent_generator, k=k, action_value_method = partial(constant_step_size, a=0.1)),
}


In [24]:

parameter_study_results = []


for name, generator in learning_agents.items():
    print(f"Doing drift problem with {name}")
    with ProcessPoolExecutor(max_workers=8) as executor:
        runs = {
            (executor.submit(k_armed_bandit_test_bed, non_stationary_bandit_generator, partial(generator, epsilon=epsilon), runs=100, steps=20_000), epsilon)
            for epsilon in epsilon_range
        }
        for run,epsilon in tqdm(runs, total=len(runs)):
            run_reward, run_optimal_action = run.result()
            parameter_study_results.append({
                "agent": name,
                "epsilon": epsilon,
                "rewards_received": run_reward,
                "optimal_actions": run_optimal_action
            })

df = pd.DataFrame(parameter_study_results)
df

Doing drift problem with increment


100%|██████████| 9/9 [01:07<00:00,  7.45s/it]

Doing drift problem with constant_step_size



100%|██████████| 9/9 [01:04<00:00,  7.14s/it]


,agent,epsilon,rewards_received,optimal_actions
0,increment,0.062500,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,..."
1,increment,0.003906,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,..."
2,increment,0.015625,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,..."
3,increment,0.250000,"[[1.093651397720687, -0.2601249455125629, 0.36...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
4,increment,0.007812,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,..."
5,increment,0.125000,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
6,increment,0.500000,"[[1.093651397720687, -0.2601249455125629, 0.36...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,..."
7,increment,0.001953,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,..."
8,increment,0.031250,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,..."
9,constant_step_size,0.031250,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,..."


In [25]:
df["average_reward"] = df["rewards_received"].apply(lambda vec: np.mean(vec[:,-100_000:], axis=0).mean())
df["prop_optimal_action"] = df["optimal_actions"].apply(lambda vec: np.mean(vec[:,-100_000:], axis=0).mean())
df

,agent,epsilon,rewards_received,optimal_actions,average_reward,prop_optimal_action
0,increment,0.062500,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,...",0.902012,0.328496
1,increment,0.003906,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,...",0.856446,0.327360
2,increment,0.015625,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,...",0.902190,0.270932
3,increment,0.250000,"[[1.093651397720687, -0.2601249455125629, 0.36...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1.078999,0.446935
4,increment,0.007812,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,...",0.949583,0.344184
5,increment,0.125000,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1.094394,0.433488
6,increment,0.500000,"[[1.093651397720687, -0.2601249455125629, 0.36...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0.504686,0.233038
7,increment,0.001953,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,...",0.849567,0.222292
8,increment,0.031250,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,...",0.990999,0.450957
9,constant_step_size,0.031250,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.232141,0.639770


In [26]:
df.query("agent == 'constant_step_size'").sort_values("epsilon")

,agent,epsilon,rewards_received,optimal_actions,average_reward,prop_optimal_action
15,constant_step_size,0.001953,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.243035,0.534017
16,constant_step_size,0.003906,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.182037,0.538180
17,constant_step_size,0.007812,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.331674,0.650057
10,constant_step_size,0.015625,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.233439,0.642964
9,constant_step_size,0.031250,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.232141,0.639770
12,constant_step_size,0.062500,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,...",1.202408,0.655431
11,constant_step_size,0.125000,"[[1.093651397720687, -0.2601249455125629, -0.7...","[[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,...",1.334587,0.692598
13,constant_step_size,0.250000,"[[1.093651397720687, -0.2601249455125629, 0.36...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1.199029,0.618246
14,constant_step_size,0.500000,"[[1.093651397720687, -0.2601249455125629, 0.36...","[[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",0.642701,0.403135
